# 1 — Named Graphs as Hyperedges in the DEIX Digital Engineering Context

This notebook establishes the pattern with a concrete engineering example
and then the next notebook builds the full translation pipeline on top of it.

## The Hyperedge Pattern

A SysML modeler holon has four named graphs:

```
<urn:holon:sysml/interior>    ← contains design model triples
<urn:holon:sysml/boundary>    ← contains SHACL shapes + portal defs
<urn:holon:sysml/projection>  ← contains outward-facing canonical data
<urn:holon:sysml/context>     ← contains membership, roles, provenance
```

The hyperedge occurs when **a graph IRI is referenced in another graph**:

- The **context** graph says the **interior** was derived from something
- The **boundary** graph defines a portal whose transform operates on the **interior**
- The **projection** was generated from the **interior** (recorded in **context**)
- A **provenance activity** in the context graph `prov:used` the interior graph IRI
  and `prov:generated` the projection graph IRI

The graph IRIs are the hyperedges.  They connect the four layers into a
self-referential structure where each layer knows about the others.

In [1]:
from holonic import Holon, TransformPortal, validate_membrane, ProvenanceTracker

## Build a SysML Tool Holon

The interior uses a **local** SysML-like vocabulary (`sysml:`).
This is NOT the DEIX vocabulary — it is what the tool natively produces.

In [2]:
sysml = Holon(
    iri="urn:holon:tool:sysml",
    label="SysML v2 Modeler",
    depth=1,

    # ── Interior: what the tool actually contains (local vocabulary) ──
    interior_ttl="""
        @prefix sysml: <urn:sysml:> .

        <urn:sysml:block:tms> a sysml:Block ;
            sysml:name       "ThermalMgmtSubsystem" ;
            sysml:stereotype "Block" ;
            sysml:package    <urn:sysml:pkg:vehicle> .

        <urn:sysml:param:tms:mass> a sysml:ValueProperty ;
            sysml:name       "mass" ;
            sysml:value      142.3 ;
            sysml:unit       "kg" ;
            sysml:owner      <urn:sysml:block:tms> .

        <urn:sysml:param:tms:power> a sysml:ValueProperty ;
            sysml:name       "maxPowerDraw" ;
            sysml:value      2.8 ;
            sysml:unit       "kW" ;
            sysml:owner      <urn:sysml:block:tms> .

        <urn:sysml:port:tms:coolant-in> a sysml:FlowPort ;
            sysml:name       "coolantInlet" ;
            sysml:direction  "in" ;
            sysml:owner      <urn:sysml:block:tms> .
    """,

    # ── Boundary: SHACL shapes for the local vocabulary ──
    boundary_ttl="""
        @prefix sysml: <urn:sysml:> .

        <urn:shapes:sysml:BlockShape> a sh:NodeShape ;
            sh:targetClass sysml:Block ;
            sh:property [
                sh:path sysml:name ;
                sh:minCount 1 ; sh:maxCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Violation ;
                sh:message "Block must have a name."
            ] .

        <urn:shapes:sysml:ValuePropertyShape> a sh:NodeShape ;
            sh:targetClass sysml:ValueProperty ;
            sh:property [
                sh:path sysml:value ;
                sh:minCount 1 ;
                sh:severity sh:Violation ;
                sh:message "ValueProperty must have a value."
            ] ;
            sh:property [
                sh:path sysml:unit ;
                sh:minCount 1 ;
                sh:datatype xsd:string ;
                sh:severity sh:Warning ;
                sh:message "ValueProperty should declare units."
            ] .
    """,

    # ── Context: holarchy membership ──
    context_ttl="""
        <urn:holon:tool:sysml> cga:memberOf <urn:holon:domain:engineering> .
    """,
)

print(sysml)

Holon('SysML v2 Modeler', depth=1, interior=25t, boundary=22t, projection=0t, context=1t)


## The Hyperedge: Graph IRIs Referencing Each Other

Now let's make the hypergraph explicit.  The context graph contains
PROV-O triples where the **graph IRIs themselves** are the subjects
and objects — not the entities inside them.

In [4]:
# The context graph references the interior and projection graph IRIs.
# This is the hyperedge pattern: a named graph (context) contains triples
# whose subjects/objects are OTHER named graph IRIs.

sysml.load_context("""
    # The interior graph is a prov:Entity (it was generated by the modeler)
    <urn:holon:tool:sysml/interior>
        a prov:Entity ;
        rdfs:label "SysML interior graph" ;
        prov:wasGeneratedBy <urn:act:sysml-model-update> ;
        prov:generatedAtTime "2026-03-15T10:00:00Z"^^xsd:dateTime .

    # The modeling activity
    <urn:act:sysml-model-update> a prov:Activity ;
        rdfs:label "SysML model update (v3.2)" ;
        prov:wasAssociatedWith <urn:agent:engineer:alice> ;
        prov:used <urn:holon:tool:sysml/interior> ;
        prov:startedAtTime "2026-03-15T09:00:00Z"^^xsd:dateTime ;
        prov:endedAtTime   "2026-03-15T10:00:00Z"^^xsd:dateTime .

    # Note: Alice would likely exist in an e.g. "Employee" interior graph
    <urn:agent:engineer:alice> a prov:Agent ;
        rdfs:label "Alice (Thermal Engineer)" .
""")

print("Context graph now references the interior graph IRI as a prov:Entity:")
print(f"  <{sysml.context.identifier}> contains triples about <{sysml.interior.identifier}>")
print()

# Show the cross-reference
for s, p, o in sorted(sysml.context):
    if "interior" in str(s) or "interior" in str(o):
        s_short = str(s).split("/")[-1]
        p_short = str(p).split("#")[-1].split("/")[-1]
        o_short = str(o)[:50]
        print(f"  {s_short:30s} {p_short:25s} {o_short}")

Context graph now references the interior graph IRI as a prov:Entity:
  <urn:holon:tool:sysml/context> contains triples about <urn:holon:tool:sysml/interior>

  urn:act:sysml-model-update     used                      urn:holon:tool:sysml/interior
  interior                       type                      http://www.w3.org/ns/prov#Entity
  interior                       label                     SysML interior graph
  interior                       generatedAtTime           2026-03-15T10:00:00+00:00
  interior                       wasGeneratedBy            urn:act:sysml-model-update


## The Boundary References the Interior Too

Portal definitions in the boundary say "this transform operates on
the source holon's interior."  The boundary graph contains the portal
IRI; the portal's `cga:sourceHolon` resolves to the holon whose
interior the CONSTRUCT query will execute against.

The boundary also references the **target** holon — which means the
boundary graph contains pointers INTO another holon entirely.

In [5]:
# Show what the boundary already contains (portals auto-registered)
print("Boundary graph contents (after Holon construction):")
print(f"  Graph IRI: {sysml.boundary.identifier}")
print(f"  Triples: {len(sysml.boundary)}")
print()
for s, p, o in sorted(sysml.boundary):
    s_short = str(s).split("/")[-1].split(":")[-1]
    p_short = str(p).split("#")[-1].split("/")[-1]
    o_short = str(o)[:60]
    print(f"  {s_short:30s} {p_short:25s} {o_short}")

Boundary graph contents (after Holon construction):
  Graph IRI: urn:holon:tool:sysml/boundary
  Triples: 22

  n4842e37ccef84fbd93fd99924e77829cb1 datatype                  http://www.w3.org/2001/XMLSchema#string
  n4842e37ccef84fbd93fd99924e77829cb1 maxCount                  1
  n4842e37ccef84fbd93fd99924e77829cb1 message                   Block must have a name.
  n4842e37ccef84fbd93fd99924e77829cb1 minCount                  1
  n4842e37ccef84fbd93fd99924e77829cb1 path                      urn:sysml:name
  n4842e37ccef84fbd93fd99924e77829cb1 severity                  http://www.w3.org/ns/shacl#Violation
  n4842e37ccef84fbd93fd99924e77829cb2 message                   ValueProperty must have a value.
  n4842e37ccef84fbd93fd99924e77829cb2 minCount                  1
  n4842e37ccef84fbd93fd99924e77829cb2 path                      urn:sysml:value
  n4842e37ccef84fbd93fd99924e77829cb2 severity                  http://www.w3.org/ns/shacl#Violation
  n4842e37ccef84fbd93fd99924e77829cb3 data

## The Projection Is Both Output and Input

When a portal transforms SysML interior data into DEIX canonical form,
the result goes into a graph that is simultaneously:

- A **projection** from the SysML holon's perspective (outward face)
- An **interior contribution** from DEIX's perspective (data it knows)

The projection graph IRI appears in:
- The SysML **context** (as `prov:generated` by the traversal activity)
- The DEIX **context** (as `prov:wasDerivedFrom` the SysML interior)

This dual participation IS the hyperedge.  The same named graph
is a container (holding projected triples) and a participant
(referenced by URI in two different context graphs).

## Summary: Where Graph IRIs Appear Across Layers

```
SysML Holon
┌──────────────────────────────────────────────────────────────┐
│ Interior <urn:holon:tool:sysml/interior>                    │
│   Contains: sysml:Block, sysml:ValueProperty triples        │
│   Referenced IN: context (as prov:Entity)                    │
│                  boundary (portal source)                    │
│                  DEIX context (as prov:wasDerivedFrom)       │
│                                                              │
│ Boundary <urn:holon:tool:sysml/boundary>                    │
│   Contains: SHACL shapes, portal definitions                 │
│   References: interior IRI (via portal sourceHolon)          │
│               DEIX holon IRI (via portal targetHolon)        │
│                                                              │
│ Projection <urn:holon:tool:sysml/projection>                │
│   Contains: DEIX-vocabulary triples (the translated data)    │
│   Referenced IN: SysML context (prov:generated)              │
│                  DEIX context (prov:wasDerivedFrom)           │
│                                                              │
│ Context <urn:holon:tool:sysml/context>                      │
│   Contains: membership, provenance activities                │
│   References: interior IRI (prov:used)                       │
│               projection IRI (prov:generated)                │
│               portal IRI (cga:viaPortal)                     │
│               DEIX interior IRI (prov:wasDerivedFrom)        │
└──────────────────────────────────────────────────────────────┘
```

Every arrow in this diagram is a triple where one named graph's IRI
appears as a subject or object in another named graph's triples.
That IS the hypergraph.  The next notebook builds the full translation
pipeline on this foundation.